# Domain-Robust Pneumonia Detection using MobileNetV2
## Multi-Dataset Research Pipeline for Edge Device Deployment

**Datasets:** Kermany · CheXpert-small · RSNA

**Research Goal:** Demonstrate cross-domain generalization challenges and evaluate simple, practical domain adaptation techniques suitable for lightweight models deployed on edge devices.

### Key Contributions:
1. Systematic cross-dataset evaluation revealing domain shift impact
2. Lightweight model (2.2M parameters) suitable for edge deployment
3. Practical unsupervised domain adaptation techniques
4. Reproducible benchmark across three diverse chest X-ray datasets

In [ ]:
import os
import sys
import random
import copy
import logging
import warnings
import tempfile
import time
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
import torchvision
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
)
from sklearn.model_selection import train_test_split

from tqdm.auto import tqdm

# Install pydicom if not available
try:
    import pydicom
    HAS_PYDICOM = True
except ImportError:
    print("Installing pydicom...")
    os.system(f"{sys.executable} -m pip install pydicom")
    import pydicom
    HAS_PYDICOM = True

warnings.filterwarnings("ignore")

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

    ## Reproducibility Setup

In [ ]:
def set_seed(seed: int):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

# Multiple seeds for statistical validity
SEEDS = [42, 123, 456]
PRIMARY_SEED = SEEDS[0]

set_seed(PRIMARY_SEED)
print(f"Reproducibility seeds: {SEEDS}")

## Logging & Device Configuration

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
print("PyTorch CUDA available:", torch.cuda.is_available())

if device.type == "cuda":
    try:
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        if hasattr(torch.cuda, "mem_get_info"):
            free_mem, total_mem = torch.cuda.mem_get_info()
            print(f"Total GPU memory: {total_mem / 1e9:.1f} GB")
    except Exception as e:
        print(f"Could not get GPU info: {e}")

## Configuration

In [ ]:
PATHS = {
    # Kermany (Mooney)
    "kermany_train": "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/train",
    "kermany_test":  "kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/test",

    # CheXpert-small
    "chexpert_root":      "/kaggle/input/datasets/ashery/chexpert",
    "chexpert_train_csv": "/kaggle/input/datasets/ashery/chexpert/train.csv",
    "chexpert_valid_csv": "/kaggle/input/datasets/ashery/chexpert/valid.csv",

    # RSNA
    "rsna_images": "/kaggle/input/datasets/abhirooppamula/rsnaaa/rsna/stage_2_train_images",
    "rsna_labels": "/kaggle/input/datasets/abhirooppamula/rsnaaa/rsna/stage_2_train_labels.csv",
}

# Model and training configuration
IMG_SIZE      = 224
BATCH_SIZE    = 32
NUM_EPOCHS    = 25
PATIENCE      = 5
LEARNING_RATE = 1e-4
NUM_WORKERS   = 0
NUM_CLASSES   = 2
CLASS_NAMES   = ["Normal", "Pneumonia"]

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print("\n" + "="*70)
print("CONFIGURATION")
print("="*70)
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {NUM_EPOCHS} (with early stopping patience={PATIENCE})")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Random seeds: {SEEDS}")

print("\nDataset paths:")
for k, v in PATHS.items():
    exists = os.path.exists(v)
    print(f" {k}: {'✓' if exists else '✗'} {v}")

## Data Transforms

In [ ]:
# Training transform with data augmentation
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Validation/test transform without augmentation
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Data transforms defined:")
print(f" Train: Resize(256) → CenterCrop({IMG_SIZE}) → Augmentation → Normalize")
print(f" Val:   Resize(256) → CenterCrop({IMG_SIZE}) → Normalize")

## Dataset Loaders
### 1. Kermany Dataset

In [ ]:
def is_valid_kermany_file(path: str) -> bool:
    """Filter out hidden files and invalid formats."""
    fname = os.path.basename(path)
    if fname.startswith("._"):
        return False
    if "__MACOSX" in path:
        return False
    return fname.lower().endswith((".png", ".jpg", ".jpeg"))

def load_kermany(transform_train=train_transform, transform_val=val_transform):
    """Load Kermany dataset using PyTorch ImageFolder."""
    train_root = PATHS["kermany_train"]
    test_root  = PATHS["kermany_test"]

    if not (os.path.isdir(train_root) and os.path.isdir(test_root)):
        raise FileNotFoundError(
            f"Kermany paths not found:\n {train_root}\n {test_root}"
        )

    train_ds = ImageFolder(
        train_root,
        transform=transform_train,
        is_valid_file=is_valid_kermany_file,
    )
    test_ds = ImageFolder(
        test_root,
        transform=transform_val,
        is_valid_file=is_valid_kermany_file,
    )

    print(f"Kermany class_to_idx: {train_ds.class_to_idx}")
    print(f" Train: {len(train_ds)} images | Test: {len(test_ds)} images")
    return train_ds, test_ds

### 2. CheXpert Dataset

In [ ]:
class CheXpertDataset(Dataset):
    """CheXpert dataset loader with frontal view filtering."""
    
    def __init__(self, csv_path, root_dir, transform=None):
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"CheXpert CSV not found: {csv_path}")
        if not os.path.exists(root_dir):
            raise FileNotFoundError(f"CheXpert root directory not found: {root_dir}")

        self.root_dir = root_dir
        self.transform = transform

        df = pd.read_csv(csv_path)

        if "Frontal/Lateral" in df.columns:
            df = df[df["Frontal/Lateral"] == "Frontal"].copy()

        if "Pneumonia" not in df.columns:
            raise KeyError("CheXpert CSV must contain a 'Pneumonia' column.")

        df = df[df["Pneumonia"].isin([0.0, 1.0])].copy()
        df["label"] = df["Pneumonia"].astype(int)

        def build_full_path(p):
            if os.path.isabs(p) and os.path.exists(p):
                return p
            p_rel = p.replace("CheXpert-v1.0-small/", "").lstrip("/\\")
            return os.path.join(self.root_dir, p_rel)

        df["full_path"] = df["Path"].apply(build_full_path)
        df = df[df["full_path"].apply(os.path.exists)].reset_index(drop=True)

        self.data = df[["full_path", "label"]].values
        print(
            f" CheXpert loaded from {os.path.basename(csv_path)}: {len(self.data)} images "
            f"(Normal: {(df['label'] == 0).sum()}, Pneumonia: {(df['label'] == 1).sum()})"
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(label)

def load_chexpert(transform_train=train_transform, transform_val=val_transform):
    """Load CheXpert dataset."""
    train_ds = CheXpertDataset(
        PATHS["chexpert_train_csv"],
        PATHS["chexpert_root"],
        transform=transform_train,
    )
    val_ds = CheXpertDataset(
        PATHS["chexpert_valid_csv"],
        PATHS["chexpert_root"],
        transform=transform_val,
    )
    return train_ds, val_ds

### 3. RSNA Dataset

In [ ]:
class RSNADataset(Dataset):
    """RSNA pneumonia detection dataset with DICOM loading."""
    
    def __init__(self, image_dir, labels_df, transform=None):
        if not os.path.exists(image_dir):
            raise FileNotFoundError(f"RSNA image directory not found: {image_dir}")
        self.image_dir = image_dir
        self.transform = transform
        self.data = labels_df.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        patient_id = row["patientId"]
        label = int(row["label"])

        dcm_path = os.path.join(self.image_dir, f"{patient_id}.dcm")
        if not os.path.exists(dcm_path):
            raise FileNotFoundError(f"DICOM file not found: {dcm_path}")

        dcm = pydicom.dcmread(dcm_path)
        pixel_array = dcm.pixel_array.astype(np.float32)

        p_min, p_max = float(pixel_array.min()), float(pixel_array.max())
        if p_max > p_min:
            pixel_array = (pixel_array - p_min) / (p_max - p_min) * 255.0
        else:
            pixel_array = np.zeros_like(pixel_array)
        pixel_array = pixel_array.astype(np.uint8)

        image = Image.fromarray(pixel_array).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

def load_rsna_labels():
    """Load RSNA labels and aggregate at patient level."""
    csv_path = PATHS["rsna_labels"]
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"RSNA labels CSV not found at {csv_path}")

    df = pd.read_csv(csv_path)
    if "patientId" not in df.columns or "Target" not in df.columns:
        raise KeyError("RSNA labels CSV must contain 'patientId' and 'Target' columns.")

    patient_labels = df.groupby("patientId")["Target"].max().reset_index()
    patient_labels["label"] = (patient_labels["Target"] > 0).astype(int)

    print(
        f" RSNA patients: {len(patient_labels)} "
        f"(Normal: {(patient_labels['label'] == 0).sum()}, "
        f"Pneumonia: {(patient_labels['label'] == 1).sum()})"
    )
    return patient_labels[["patientId", "label"]]

def load_rsna(transform_train=train_transform, transform_val=val_transform, test_ratio=0.2):
    """Load RSNA dataset with train/test split."""
    labels_df = load_rsna_labels()

    train_df, test_df = train_test_split(
        labels_df,
        test_size=test_ratio,
        stratify=labels_df["label"],
        random_state=PRIMARY_SEED,
    )
    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    train_ds = RSNADataset(PATHS["rsna_images"], train_df, transform=transform_train)
    test_ds  = RSNADataset(PATHS["rsna_images"], test_df,  transform=transform_val)

    print(f" RSNA Train: {len(train_ds)} | Test: {len(test_ds)}")
    return train_ds, test_ds

print("Dataset loaders defined successfully.")

## Load All Datasets

In [ ]:
datasets = {}

print("\n" + "="*70)
print("LOADING DATASETS")
print("="*70)

try:
    print("\nLoading Kermany dataset...")
    kermany_train, kermany_test = load_kermany()
    datasets["Kermany"] = {"train": kermany_train, "test": kermany_test}
except Exception as e:
    print(f"[Warning] Skipping Kermany: {e}")

try:
    print("\nLoading CheXpert dataset...")
    chexpert_train, chexpert_test = load_chexpert()
    datasets["CheXpert"] = {"train": chexpert_train, "test": chexpert_test}
except Exception as e:
    print(f"[Warning] Skipping CheXpert: {e}")

try:
    print("\nLoading RSNA dataset...")
    rsna_train, rsna_test = load_rsna()
    datasets["RSNA"] = {"train": rsna_train, "test": rsna_test}
except Exception as e:
    print(f"[Warning] Skipping RSNA: {e}")

if not datasets:
    raise RuntimeError("No datasets were successfully loaded. Please check paths and files.")

print("\n" + "="*70)
print("DATASET SUMMARY")
print("="*70)
for name, ds in datasets.items():
    print(f" {name:12s}: Train={len(ds['train']):6d}, Test={len(ds['test']):5d}")
print("="*70)

## Utility Functions

In [ ]:
def compute_class_weights(dataset):
    """Compute class weights for handling imbalanced datasets."""
    labels = []
    for _, y in dataset:
        labels.append(int(y))
    labels = np.array(labels)
    counts = np.bincount(labels, minlength=2)
    weights = counts.sum() / (counts + 1e-6)
    weights = torch.tensor(weights, dtype=torch.float32, device=device)
    return weights

def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    """Plot confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )
    plt.title(title, fontsize=12, fontweight='bold')
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.show()

print("Utility functions defined.")

## Model Architecture: MobileNetV2

In [ ]:
def create_model(num_classes=NUM_CLASSES, pretrained=True):
    """
    Create MobileNetV2 model with custom classifier.
    Optimized for edge deployment.
    """
    if pretrained:
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    else:
        model = models.mobilenet_v2(weights=None)

    # Freeze feature extractor
    for param in model.features.parameters():
        param.requires_grad = False

    # Custom classifier head
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(1280, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes),
    )
    
    model = model.to(device, non_blocking=True)
    return model

# Display architecture
_model_test = create_model()
print("\n" + "="*70)
print("MODEL ARCHITECTURE: MobileNetV2")
print("="*70)
total_params = sum(p.numel() for p in _model_test.parameters())
trainable_params = sum(p.numel() for p in _model_test.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total parameters:     {total_params:,} ({total_params/1e6:.2f}M)")
print(f"Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
print(f"Frozen parameters:    {frozen_params:,} ({frozen_params/1e6:.2f}M)")
print("\nStrategy: Transfer learning with frozen features")
print("="*70)
del _model_test

## Baseline Model: ResNet50

In [ ]:
def create_resnet50(num_classes=NUM_CLASSES, pretrained=True):
    """Create ResNet50 baseline for comparison."""
    if pretrained:
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    else:
        model = models.resnet50(weights=None)
    
    # Freeze feature extractor
    for param in model.parameters():
        param.requires_grad = False
    
    # Replace final layer
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(num_ftrs, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes),
    )
    
    model = model.to(device, non_blocking=True)
    return model

_resnet_test = create_resnet50()
resnet_total = sum(p.numel() for p in _resnet_test.parameters())
resnet_trainable = sum(p.numel() for p in _resnet_test.parameters() if p.requires_grad)
print(f"\nResNet50 (Baseline): {resnet_total:,} total params ({resnet_total/1e6:.2f}M)")
print(f"                     {resnet_trainable:,} trainable params ({resnet_trainable/1e6:.2f}M)")
del _resnet_test

## Training Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, desc="Train"):
    """Train for one epoch."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, desc=desc, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, desc="Val"):
    """Evaluate model on a dataset."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_labels, all_preds, all_probs = [], [], []

    for images, labels in tqdm(loader, desc=desc, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

    return (
        running_loss / total,
        correct / total,
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
    )

def compute_metrics(y_true, y_pred, y_prob):
    """Compute comprehensive evaluation metrics."""
    metrics = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1-Score":  f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.0,
    }
    return metrics

def train_model(
    model,
    train_loader,
    val_loader,
    num_epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    class_weights=None,
    patience=PATIENCE,
):
    """Train model with early stopping."""
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), 
        lr=lr
    )
    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=max(num_epochs // 3, 1),
        gamma=0.1,
    )

    best_val_acc = 0.0
    best_model_state = None
    patience_counter = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer,
            desc=f"Epoch {epoch+1}/{num_epochs}",
        )
        val_loss, val_acc, _, _, _ = evaluate(
            model, val_loader, criterion,
            desc=f"Val {epoch+1}/{num_epochs}",
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f" Epoch {epoch+1:2d}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print(f"  ✓ New best validation accuracy: {best_val_acc:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping triggered at epoch {epoch+1}")
                break

        scheduler.step()

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    print(f" ✓ Training complete. Best Val Accuracy: {best_val_acc:.4f}")
    return model, history

print("Training functions defined.")

## Domain Adaptation Techniques

In [ ]:
@torch.no_grad()
def adaptive_bn_test_time(model, loader):
    """
    UNSUPERVISED Test-Time Batch Normalization Adaptation.
    
    Key: Uses ONLY test images (no labels required).
    Updates BN running statistics without changing weights.
    Practical for deployment - can adapt on unlabeled target data.
    """
    model.train()  # Set to train mode to update BN statistics
    
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.momentum = 0.1
    
    for images, _ in loader:
        images = images.to(device, non_blocking=True)
        _ = model(images)
    
    model.eval()
    return model

@torch.no_grad()
def test_time_augmentation(model, loader, criterion, n_aug=5):
    """
    Test-Time Augmentation (TTA).
    Apply augmented versions of each test image and average predictions.
    """
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    running_loss, total = 0.0, 0
    
    for images, labels in tqdm(loader, desc="TTA Evaluation", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        outputs_list = [model(images)]
        
        for _ in range(n_aug - 1):
            aug_images = torch.flip(images, dims=[3])  # Flip width
            outputs_list.append(model(aug_images))
        
        outputs = torch.mean(torch.stack(outputs_list), dim=0)
        loss = criterion(outputs, labels)
        
        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())
    
    return (
        running_loss / total,
        np.mean(np.array(all_labels) == np.array(all_preds)),
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
    )

print("Domain adaptation techniques defined.")

## Experiment 1: Independent Training (Same-Domain Performance)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 1: INDEPENDENT TRAINING (Same-Domain Performance)")
print("="*70)
print("Goal: Establish baseline performance when train/test come from same dataset")
print("="*70)

independent_results_all_seeds = defaultdict(list)
trained_models = {}  # Store models from primary seed

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'─'*70}")
    print(f"RUN {seed_idx + 1}/3 (Seed: {seed})")
    print(f"{'─'*70}")
    
    set_seed(seed)
    
    for ds_name, ds_data in datasets.items():
        print(f"\nTraining on {ds_name}...")
        
        model = create_model()
        
        train_loader = DataLoader(
            ds_data["train"],
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
        )
        test_loader = DataLoader(
            ds_data["test"],
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
        )
        
        class_weights = compute_class_weights(ds_data["train"])
        model, history = train_model(
            model, train_loader, test_loader,
            num_epochs=NUM_EPOCHS,
            lr=LEARNING_RATE,
            class_weights=class_weights,
            patience=PATIENCE,
        )
        
        criterion = nn.CrossEntropyLoss()
        _, _, y_true, y_pred, y_prob = evaluate(
            model, test_loader, criterion, desc=f"{ds_name} Test",
        )
        metrics = compute_metrics(y_true, y_pred, y_prob)
        
        independent_results_all_seeds[ds_name].append(metrics)
        
        if seed == PRIMARY_SEED:
            trained_models[ds_name] = model
        
        print(f"\n✓ {ds_name} (Seed {seed}) Results:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")

# Aggregate results
print("\n" + "="*70)
print("EXPERIMENT 1 SUMMARY (Mean ± Std across 3 seeds)")
print("="*70)

independent_summary = {}
for ds_name, results_list in independent_results_all_seeds.items():
    metric_names = results_list[0].keys()
    summary = {}
    for metric in metric_names:
        values = [r[metric] for r in results_list]
        summary[f"{metric}_mean"] = np.mean(values)
        summary[f"{metric}_std"] = np.std(values)
    independent_summary[ds_name] = summary

for ds_name, summary in independent_summary.items():
    print(f"\n{ds_name}:")
    for metric in ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]:
        mean = summary[f"{metric}_mean"]
        std = summary[f"{metric}_std"]
        print(f"  {metric:12s}: {mean:.4f} ± {std:.4f}")

print("\n" + "="*70)

## Experiment 2: Cross-Dataset Evaluation (Domain Shift Problem)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 2: CROSS-DATASET EVALUATION (Domain Shift Problem)")
print("="*70)
print("Goal: Quantify performance drop when testing on different datasets")
print("="*70)

cross_matrix = {}
cross_detailed_results = {}
criterion = nn.CrossEntropyLoss()

for train_name, model in trained_models.items():
    cross_matrix[train_name] = {}
    cross_detailed_results[train_name] = {}
    
    for test_name, ds_data in datasets.items():
        test_loader = DataLoader(
            ds_data["test"],
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
        )
        
        _, _, y_true, y_pred, y_prob = evaluate(
            model, test_loader, criterion, desc=f"{train_name}→{test_name}",
        )
        metrics = compute_metrics(y_true, y_pred, y_prob)
        
        cross_matrix[train_name][test_name] = metrics["Accuracy"]
        cross_detailed_results[train_name][test_name] = metrics
        
        if train_name != test_name:
            print(
                f" Train {train_name:10s} → Test {test_name:10s} | "
                f"Acc: {metrics['Accuracy']:.4f} | "
                f"F1: {metrics['F1-Score']:.4f}"
            )

cross_df = pd.DataFrame(cross_matrix).T
cross_df.index.name = "Trained On"
cross_df.columns.name = "Tested On"

print("\n" + "="*70)
print("CROSS-DATASET GENERALIZATION MATRIX (Accuracy)")
print("="*70)
print(cross_df.round(4).to_string())
print("\nNote: Diagonal = same-domain, Off-diagonal = cross-domain")
print("="*70)

In [ ]:
print("\n" + "="*70)
print("CROSS-DOMAIN PERFORMANCE DROP ANALYSIS")
print("="*70)

drop_analysis = []

for train_ds in cross_df.index:
    same_domain = cross_df.loc[train_ds, train_ds]
    cross_domain_scores = [
        cross_df.loc[train_ds, test_ds] 
        for test_ds in cross_df.columns 
        if test_ds != train_ds
    ]
    avg_cross = np.mean(cross_domain_scores)
    drop_abs = same_domain - avg_cross
    drop_pct = (drop_abs / same_domain) * 100
    
    drop_analysis.append({
        "Dataset": train_ds,
        "Same-Domain": same_domain,
        "Cross-Domain Avg": avg_cross,
        "Drop (Absolute)": drop_abs,
        "Drop (%)": drop_pct,
    })
    
    print(f"{train_ds}:")
    print(f"  Same-domain accuracy:     {same_domain:.4f}")
    print(f"  Cross-domain avg:         {avg_cross:.4f}")
    print(f"  Performance drop:         {drop_abs:.4f} ({drop_pct:.1f}%)")
    print()

drop_df = pd.DataFrame(drop_analysis)
avg_drop_pct = drop_df["Drop (%)"].mean()
print(f"\n✓ Average cross-domain performance drop: {avg_drop_pct:.1f}%")
print("="*70)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    cross_df, 
    annot=True, 
    fmt='.3f', 
    cmap='RdYlGn',
    vmin=0.5, 
    vmax=1.0,
    cbar_kws={'label': 'Accuracy'},
    linewidths=0.5,
    linecolor='gray',
)
plt.title(
    'Cross-Dataset Generalization Matrix\n'
    '(Diagonal = Same-Domain, Off-Diagonal = Cross-Domain)',
    fontsize=14, fontweight='bold', pad=20,
)
plt.ylabel('Trained On', fontsize=12, fontweight='bold')
plt.xlabel('Tested On', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cross_dataset_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## Experiment 3: Mixed Training (Multi-Source Domain Generalization)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 3: MIXED TRAINING (Multi-Source Domain Generalization)")
print("="*70)

mixed_results_all_seeds = defaultdict(list)

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'─'*70}")
    print(f"RUN {seed_idx + 1}/3 (Seed: {seed})")
    print(f"{'─'*70}")
    
    set_seed(seed)
    
    mixed_full = ConcatDataset([ds["train"] for ds in datasets.values()])
    print(f"Combined training set: {len(mixed_full)} images")
    
    val_ratio = 0.1
    val_size = int(len(mixed_full) * val_ratio)
    train_size = len(mixed_full) - val_size
    
    mixed_train_ds, mixed_val_ds = random_split(
        mixed_full, [train_size, val_size],
        generator=torch.Generator().manual_seed(seed),
    )
    
    mixed_train_loader = DataLoader(
        mixed_train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    )
    mixed_val_loader = DataLoader(
        mixed_val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    )
    
    mixed_model = create_model()
    class_weights_mixed = compute_class_weights(mixed_train_ds)
    mixed_model, mixed_history = train_model(
        mixed_model, mixed_train_loader, mixed_val_loader,
        num_epochs=NUM_EPOCHS, lr=LEARNING_RATE,
        class_weights=class_weights_mixed, patience=PATIENCE,
    )
    
    criterion = nn.CrossEntropyLoss()
    for test_name, ds_data in datasets.items():
        test_loader = DataLoader(
            ds_data["test"], batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
        )
        _, _, y_true, y_pred, y_prob = evaluate(
            mixed_model, test_loader, criterion, desc=f"Mixed→{test_name}",
        )
        metrics = compute_metrics(y_true, y_pred, y_prob)
        mixed_results_all_seeds[test_name].append(metrics)
        print(
            f"Mixed model on {test_name}: "
            f"Acc={metrics['Accuracy']:.4f}, "
            f"F1={metrics['F1-Score']:.4f}"
        )

print("\n" + "="*70)
print("EXPERIMENT 3 SUMMARY (Mean ± Std across 3 seeds)")
print("="*70)

mixed_summary = {}
for test_name, results_list in mixed_results_all_seeds.items():
    summary = {}
    for metric in ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]:
        values = [r[metric] for r in results_list]
        summary[f"{metric}_mean"] = np.mean(values)
        summary[f"{metric}_std"] = np.std(values)
    mixed_summary[test_name] = summary

for test_name, summary in mixed_summary.items():
    print(f"\n{test_name}:")
    for metric in ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]:
        mean = summary[f"{metric}_mean"]
        std = summary[f"{metric}_std"]
        print(f"  {metric:12s}: {mean:.4f} ± {std:.4f}")

print("\n" + "="*70)

## Experiment 4: Unsupervised Domain Adaptation

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 4: UNSUPERVISED DOMAIN ADAPTATION")
print("="*70)
print("CRITICAL: No target training labels used - simulates real deployment")
print("="*70)

domain_adapt_results = []
criterion = nn.CrossEntropyLoss()

for train_name, base_model in trained_models.items():
    for test_name, ds_data in datasets.items():
        if train_name == test_name:
            continue
        
        print(f"\n{'─'*60}")
        print(f"Train: {train_name} → Test: {test_name}")
        print(f"{'─'*60}")
        
        test_loader = DataLoader(
            ds_data["test"], batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
        )
        
        # 1. BASELINE (no adaptation)
        model_baseline = copy.deepcopy(base_model)
        _, _, y_true, y_pred_base, y_prob_base = evaluate(
            model_baseline, test_loader, criterion, desc="Baseline",
        )
        metrics_baseline = compute_metrics(y_true, y_pred_base, y_prob_base)
        
        # 2. ADAPTIVE BATCH NORMALIZATION
        model_adapted_bn = copy.deepcopy(base_model)
        adaptive_bn_test_time(model_adapted_bn, test_loader)
        _, _, _, y_pred_bn, y_prob_bn = evaluate(
            model_adapted_bn, test_loader, criterion, desc="Adaptive BN",
        )
        metrics_bn = compute_metrics(y_true, y_pred_bn, y_prob_bn)
        
        # 3. TEST-TIME AUGMENTATION
        model_tta = copy.deepcopy(base_model)
        _, _, _, y_pred_tta, y_prob_tta = test_time_augmentation(
            model_tta, test_loader, criterion, n_aug=5,
        )
        metrics_tta = compute_metrics(y_true, y_pred_tta, y_prob_tta)
        
        domain_adapt_results.append({
            "Train": train_name,
            "Test": test_name,
            "Baseline": metrics_baseline["Accuracy"],
            "Adaptive BN": metrics_bn["Accuracy"],
            "TTA": metrics_tta["Accuracy"],
            "Improvement (BN)": metrics_bn["Accuracy"] - metrics_baseline["Accuracy"],
            "Improvement (TTA)": metrics_tta["Accuracy"] - metrics_baseline["Accuracy"],
        })
        
        print(f"Baseline:          Acc={metrics_baseline['Accuracy']:.4f}")
        print(f"Adaptive BN:       Acc={metrics_bn['Accuracy']:.4f} "
              f"(+{metrics_bn['Accuracy'] - metrics_baseline['Accuracy']:.4f})")
        print(f"TTA:               Acc={metrics_tta['Accuracy']:.4f} "
              f"(+{metrics_tta['Accuracy'] - metrics_baseline['Accuracy']:.4f})")

da_df = pd.DataFrame(domain_adapt_results)
print("\n" + "="*70)
print("DOMAIN ADAPTATION RESULTS SUMMARY")
print("="*70)
print(da_df.round(4).to_string(index=False))

avg_imp_bn = da_df["Improvement (BN)"].mean()
avg_imp_tta = da_df["Improvement (TTA)"].mean()
print(f"\nAverage improvement with Adaptive BN:  +{avg_imp_bn:.4f}")
print(f"Average improvement with TTA:          +{avg_imp_tta:.4f}")
print("="*70)

In [ ]:
methods = ['Baseline\n(No Adapt)', 'Adaptive BN', 'Test-Time\nAugmentation']
accuracies = [
    da_df['Baseline'].mean(),
    da_df['Adaptive BN'].mean(),
    da_df['TTA'].mean(),
]
colors = ['#e74c3c', '#f39c12', '#2ecc71']

plt.figure(figsize=(10, 6))
bars = plt.bar(methods, accuracies, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2., height,
        f'{acc:.4f}',
        ha='center', va='bottom', fontweight='bold', fontsize=11
    )

same_domain_avg = np.mean([cross_df.loc[ds, ds] for ds in cross_df.index])
plt.axhline(
    y=same_domain_avg, color='blue', linestyle='--', linewidth=2,
    label=f'Same-Domain Avg ({same_domain_avg:.3f})'
)

plt.ylabel('Cross-Domain Accuracy', fontsize=12, fontweight='bold')
plt.title(
    'Effectiveness of Unsupervised Domain Adaptation Techniques\n'
    '(Average across all cross-domain scenarios)',
    fontsize=14, fontweight='bold', pad=20
)
plt.ylim(0.5, 1.0)
plt.legend(fontsize=10, loc='lower right')
plt.grid(axis='y', alpha=0.3, linestyle=':')
plt.tight_layout()
plt.savefig('domain_adaptation_effectiveness.png', dpi=300, bbox_inches='tight')
plt.show()

## Baseline Comparison: MobileNetV2 vs ResNet50

In [ ]:
print("\n" + "="*70)
print("BASELINE COMPARISON: Lightweight vs Standard Architecture")
print("="*70)

comparison_dataset = list(datasets.keys())[0]
print(f"\nTraining both models on {comparison_dataset}...\n")

set_seed(PRIMARY_SEED)
ds_data = datasets[comparison_dataset]

train_loader = DataLoader(
    ds_data["train"], batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
)
test_loader = DataLoader(
    ds_data["test"], batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
)

class_weights = compute_class_weights(ds_data["train"])
criterion = nn.CrossEntropyLoss()

print("Training ResNet50 (heavier baseline)...")
resnet_model = create_resnet50()
resnet_model, _ = train_model(
    resnet_model, train_loader, test_loader,
    num_epochs=NUM_EPOCHS, lr=LEARNING_RATE,
    class_weights=class_weights, patience=PATIENCE,
)

_, _, y_true_r, y_pred_r, y_prob_r = evaluate(
    resnet_model, test_loader, criterion, desc="ResNet50 Test"
)
resnet_metrics = compute_metrics(y_true_r, y_pred_r, y_prob_r)

mobilenet_metrics = independent_results_all_seeds[comparison_dataset][0]
mobilenet_params = 2.2
resnet_params = 25.6

print("\n" + "="*70)
print(f"ARCHITECTURE COMPARISON ON {comparison_dataset}")
print("="*70)
print(f"\nMobileNetV2 (Ours):")
print(f"  Parameters:  {mobilenet_params:.1f}M")
print(f"  Accuracy:    {mobilenet_metrics['Accuracy']:.4f}")
print(f"  F1-Score:    {mobilenet_metrics['F1-Score']:.4f}")

print(f"\nResNet50 (Baseline):")
print(f"  Parameters:  {resnet_params:.1f}M")
print(f"  Accuracy:    {resnet_metrics['Accuracy']:.4f}")
print(f"  F1-Score:    {resnet_metrics['F1-Score']:.4f}")

acc_diff = mobilenet_metrics['Accuracy'] - resnet_metrics['Accuracy']
param_ratio = resnet_params / mobilenet_params

print(f"\nComparison:")
print(f"  Accuracy difference:  {acc_diff:+.4f}")
print(f"  Parameter ratio:      {param_ratio:.1f}x")
print(f"\n✓ MobileNetV2 achieves comparable accuracy with {param_ratio:.1f}x fewer parameters!")
print("="*70)

## Deployment Analysis

In [ ]:
print("\n" + "="*70)
print("MODEL SIZE & DEPLOYMENT FEASIBILITY ANALYSIS")
print("="*70)

model = create_model()

# 1. Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"1. PARAMETER COUNT")
print(f"   Total parameters:     {total_params:,} ({total_params/1e6:.2f}M)")
print(f"   Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")

# 2. Model size on disk
temp_path = tempfile.mktemp(suffix='.pth')
torch.save(model.state_dict(), temp_path)
model_size_mb = os.path.getsize(temp_path) / (1024**2)
os.remove(temp_path)
print(f"\n2. STORAGE REQUIREMENTS")
print(f"   Model file size: {model_size_mb:.2f} MB")

# 3. Inference time
model.eval()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
print(f"\n3. INFERENCE SPEED")

for _ in range(20):
    with torch.no_grad():
        _ = model(dummy_input)
if device.type == 'cuda':
    torch.cuda.synchronize()

times = []
for _ in range(100):
    start = time.time()
    with torch.no_grad():
        _ = model(dummy_input)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    times.append(time.time() - start)

avg_time_ms = np.mean(times) * 1000
std_time_ms = np.std(times) * 1000
throughput = 1 / np.mean(times)

print(f"   Device: {device}")
print(f"   Average inference time: {avg_time_ms:.2f} ± {std_time_ms:.2f} ms")
print(f"   Throughput: {throughput:.1f} images/second")

# 4. Memory usage
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = model(dummy_input)
    peak_mem_mb = torch.cuda.max_memory_allocated() / (1024**2)
    print(f"\n4. MEMORY FOOTPRINT")
    print(f"   Peak GPU memory: {peak_mem_mb:.2f} MB")

print("\n" + "="*70)
print("DEPLOYMENT VERDICT")
print("="*70)
print(f"✓ Model size:        {model_size_mb:.2f} MB ")
print(f"✓ Inference speed:   {avg_time_ms:.2f} ms ")
print(f"✓ Parameter count:   {total_params/1e6:.2f}M ")
print("="*70)

## Final Comprehensive Summary

In [ ]:
print("\n" + "="*70)
print("FINAL RESEARCH SUMMARY")
print("="*70)

print("\n" + "─"*70)
print("1. SAME-DOMAIN PERFORMANCE")
print("─"*70)
for ds_name, summary in independent_summary.items():
    acc_mean = summary["Accuracy_mean"]
    acc_std = summary["Accuracy_std"]
    print(f"{ds_name:12s}: {acc_mean:.4f} ± {acc_std:.4f}")

print("\n" + "─"*70)
print("2. CROSS-DOMAIN GENERALIZATION MATRIX")
print("─"*70)
print(cross_df.round(4).to_string())

print("\n" + "─"*70)
print("3. PERFORMANCE DROP ANALYSIS")
print("─"*70)
print(drop_df[["Dataset", "Same-Domain", "Cross-Domain Avg", "Drop (%)"]].round(4).to_string(index=False))
print(f"\nAverage cross-domain drop: {avg_drop_pct:.1f}%")

print("\n" + "─"*70)
print("4. MIXED TRAINING RESULTS")
print("─"*70)
for test_name, summary in mixed_summary.items():
    acc_mean = summary["Accuracy_mean"]
    acc_std = summary["Accuracy_std"]
    print(f"{test_name:12s}: {acc_mean:.4f} ± {acc_std:.4f}")

print("\n" + "─"*70)
print("5. DOMAIN ADAPTATION IMPROVEMENTS")
print("─"*70)
print(f"Average improvement with Adaptive BN:  +{avg_imp_bn:.4f}")
print(f"Average improvement with TTA:          +{avg_imp_tta:.4f}")

print("\n" + "─"*70)
print("6. DEPLOYMENT CHARACTERISTICS")
print("─"*70)
print(f"Model:              MobileNetV2")
print(f"Parameters:         {total_params/1e6:.2f}M")
print(f"Model size:         {model_size_mb:.2f} MB")
print(f"Inference time:     {avg_time_ms:.2f} ms")

print("\n" + "="*70)


print("\n" + "="*70)
print("✓ ALL EXPERIMENTS COMPLETED SUCCESSFULLY")

print("="*70)